# Dynamic vs Fixed/Variable Electricity Cost Comparison
Interactive dashboard comparing hourly dynamic spot-price costs with fixed/variable contract costs

## 1. Import Required Libraries and Load Data

In [10]:
import json
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, HTML
import warnings
from datetime import datetime, timedelta
import pyarrow.parquet as pq

warnings.filterwarnings('ignore')

# Constants
VAT_RATE = 0.21
ELEC_TAX = 0.11085  # €/kWh
PIEK_HOURS = list(range(7, 23))  # 07:00 - 23:00
DAL_HOURS = list(range(23, 24)) + list(range(0, 7))  # 23:00 - 07:00

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


## 2. Load and Prepare ENTSOE Spot Price Data

In [11]:
# Load ENTSOE spot prices
entsoe_parquet_path = r'c:\Users\20203525\Documents\2025 2026\WB4U\wb4y-webscraper\Visualizations\entsoe_prices_NL_2025_2026_combined.parquet'

try:
    # Check if file exists first
    import os
    if not os.path.exists(entsoe_parquet_path):
        print(f"✗ File not found: {entsoe_parquet_path}")
        df_entsoe = None
    else:
        df_entsoe = pd.read_parquet(entsoe_parquet_path)
        print(f"✓ ENTSOE data loaded: {len(df_entsoe)} rows")
        print(f"  Columns: {df_entsoe.columns.tolist()}")
        print(f"  Index type: {type(df_entsoe.index)}")
        
        # Identify timestamp and price columns
        ts_col = None
        price_col = None
        
        # Try common column names
        for col in ['ts_utc', 'timestamp', 'time', 'date']:
            if col in df_entsoe.columns:
                ts_col = col
                break
        
        for col in ['price', 'Price', 'price_eur_per_mwh', 'price_eur_per_kwh']:
            if col in df_entsoe.columns:
                price_col = col
                break
        
        if ts_col is None or price_col is None:
            print(f"✗ Could not identify timestamp or price columns")
            print(f"  Looking for: timestamp column in {['ts_utc', 'timestamp', 'time', 'date']}")
            print(f"  Looking for: price column in {['price', 'Price', 'price_eur_per_mwh', 'price_eur_per_kwh']}")
            df_entsoe = None
        else:
            # Set proper index
            df_entsoe[ts_col] = pd.to_datetime(df_entsoe[ts_col])
            df_entsoe = df_entsoe.set_index(ts_col)
            
            print(f"  Using timestamp column: {ts_col}")
            print(f"  Using price column: {price_col}")
            print(f"  Date range: {df_entsoe.index.min()} to {df_entsoe.index.max()}")
            
            # Rename or create price_eur_per_kwh column
            # Assuming price column is already in €/kWh or is in €/MWh
            if 'price_eur_per_mwh' in df_entsoe.columns or (price_col == 'price' and df_entsoe[price_col].max() > 100):
                # Likely in €/MWh, convert to €/kWh
                df_entsoe['price_eur_per_kwh'] = df_entsoe[price_col] / 1000
                print("  Converted price from €/MWh to €/kWh")
            else:
                # Already in €/kWh
                df_entsoe['price_eur_per_kwh'] = df_entsoe[price_col]
                print("  Price already in €/kWh")
            
            print(f"✓ Spot prices ready")
            print(f"  Price range: €{df_entsoe['price_eur_per_kwh'].min():.4f} - €{df_entsoe['price_eur_per_kwh'].max():.4f}/kWh")
    
except Exception as e:
    import traceback
    print(f"✗ Error loading ENTSOE data: {e}")
    print(f"  Traceback: {traceback.format_exc()}")
    df_entsoe = None

✓ ENTSOE data loaded: 38388 rows
  Columns: ['ts_utc', 'price']
  Index type: <class 'pandas.core.indexes.range.RangeIndex'>
  Using timestamp column: ts_utc
  Using price column: price
  Date range: 2025-01-01 00:15:00+00:00 to 2026-02-04 23:45:00+00:00
  Price already in €/kWh
✓ Spot prices ready
  Price range: €-0.3500 - €0.5235/kWh


## 3. Load Provider Tariffs and Calculate Dynamic Costs

In [12]:
import sys
from pathlib import Path
sys.path.insert(0, r'c:\Users\20203525\Documents\2025 2026\WB4U\wb4y-webscraper\Visualizations')

try:
    from load_tariffs import load_provider_tariffs, normalize_name
    print("✓ load_tariffs module imported")
except ImportError as e:
    print(f"✗ Could not import load_tariffs: {e}")

# Load provider tariffs
tariff_xlsx_path = Path(r'c:\Users\20203525\Documents\2025 2026\WB4U\wb4y-webscraper\Visualizations\provider_tariffs.xlsx')

try:
    if not tariff_xlsx_path.exists():
        print(f"✗ Provider tariffs file not found: {tariff_xlsx_path}")
        dynamic_providers = {}
    else:
        tariff_book = load_provider_tariffs(tariff_xlsx_path)
        dynamic_providers = {}
        
        for provider_key, contracts in tariff_book.providers.items():
            for contract_type, tariff in contracts.items():
                if contract_type == 'dynamic':
                    if provider_key not in dynamic_providers:
                        dynamic_providers[provider_key] = {}
                    dynamic_providers[provider_key]['margin'] = tariff.price_eur_per_kwh
                    dynamic_providers[provider_key]['provider_name'] = tariff.provider
        
        print(f"✓ Provider tariffs loaded: {len(dynamic_providers)} providers with dynamic contracts")
        for provider_key, data in list(dynamic_providers.items())[:5]:
            print(f"  {data['provider_name']}: €{data['margin']:.6f}/kWh margin")
        
except Exception as e:
    import traceback
    print(f"✗ Error loading provider tariffs: {e}")
    print(f"  Traceback: {traceback.format_exc()}")
    dynamic_providers = {}

✓ load_tariffs module imported
✓ Provider tariffs loaded: 19 providers with dynamic contracts
  Mega Energie: €0.018150/kWh margin
  Greenchoice: €0.033880/kWh margin
  Vandebron: €0.025710/kWh margin
  Budget Energie: €0.020990/kWh margin
  Oxxio: €0.018490/kWh margin


In [13]:
# Calculate dynamic costs for each provider
def calculate_dynamic_costs(df_entsoe, dynamic_providers):
    """Calculate hourly dynamic costs per provider"""
    if df_entsoe is None:
        print("✗ Cannot calculate dynamic costs: ENTSOE data is None")
        return {}
    
    if not dynamic_providers:
        print("✗ No dynamic providers found")
        return {}
    
    dynamic_costs = {}
    
    for provider_key, provider_data in dynamic_providers.items():
        provider_name = provider_data['provider_name']
        margin = provider_data['margin']
        
        # Formula: (spot_price * (1 + VAT_RATE) + margin + elec_tax)
        cost_per_kwh = (df_entsoe['price_eur_per_kwh'] * (1 + VAT_RATE) + margin + ELEC_TAX)
        
        df_provider = pd.DataFrame({
            'timestamp': df_entsoe.index,
            'spot_price_eur_per_kwh': df_entsoe['price_eur_per_kwh'],
            'total_cost_eur_per_kwh': cost_per_kwh,
            'provider': provider_name,
            'margin': margin
        })
        
        dynamic_costs[provider_name] = df_provider
    
    return dynamic_costs

dynamic_costs = calculate_dynamic_costs(df_entsoe, dynamic_providers)

if not dynamic_costs:
    print("✗ No dynamic costs were calculated. Check:")
    print("  - ENTSOE data loaded (Cell 2 status)")
    print("  - Provider tariffs loaded with 'dynamic' contracts (Cell 3 status)")
else:
    print(f"✓ Dynamic costs calculated for {len(dynamic_costs)} providers")

    if dynamic_costs:
        first_provider = list(dynamic_costs.keys())[0]
        print(f"  Example ({first_provider}):")
        print(f"    Min cost: €{dynamic_costs[first_provider]['total_cost_eur_per_kwh'].min():.4f}/kWh")
        print(f"    Max cost: €{dynamic_costs[first_provider]['total_cost_eur_per_kwh'].max():.4f}/kWh")
        print(f"    Avg cost: €{dynamic_costs[first_provider]['total_cost_eur_per_kwh'].mean():.4f}/kWh")

✓ Dynamic costs calculated for 19 providers
  Example (Mega Energie):
    Min cost: €-0.2945/kWh
    Max cost: €0.7624/kWh
    Avg cost: €0.2367/kWh


## 4. Load Fixed/Variable Contract Data

In [ ]:
# Discover and load contract data from all available Parquet files
from pathlib import Path
import glob

parquet_base_dir = Path(r'c:\Users\20203525\Documents\2025 2026\WB4U\wb4y-webscraper\Attempt3\parquet_tariffs')

# Discover all available provider_id/year combinations and extract provider names
provider_year_map = {}  # {provider_id: {year: path}}
provider_id_to_names = {}  # {provider_id: [list of provider names]}

if parquet_base_dir.exists():
    for provider_dir in parquet_base_dir.glob('provider_id=*'):
        provider_str = provider_dir.name
        try:
            provider_id = int(provider_str.split('=')[1])
            if provider_id not in provider_year_map:
                provider_year_map[provider_id] = {}
            if provider_id not in provider_id_to_names:
                provider_id_to_names[provider_id] = set()
            
            for year_dir in provider_dir.glob('year=*'):
                year_str = year_dir.name
                try:
                    year = int(year_str.split('=')[1])
                    parquet_file = year_dir / 'part-0.parquet'
                    if parquet_file.exists():
                        # Read parquet to extract provider name
                        try:
                            df_temp = pd.read_parquet(parquet_file)
                            if not df_temp.empty and 'provider' in df_temp.columns:
                                # Get unique provider names from this parquet
                                provider_names = df_temp['provider'].unique()
                                for pname in provider_names:
                                    if pname and str(pname).strip():
                                        provider_id_to_names[provider_id].add(str(pname).strip())
                        except:
                            pass
                        
                        provider_year_map[provider_id][year] = str(parquet_file)
                except (IndexError, ValueError):
                    pass
        except (IndexError, ValueError):
            pass

# Create a sorted list of provider options
provider_options = []
for provider_id in sorted(provider_year_map.keys()):
    names = sorted(provider_id_to_names[provider_id]) if provider_id_to_names[provider_id] else [f"Provider {provider_id}"]
    provider_name = names[0]  # Use first name
    provider_options.append((provider_name, provider_id))

print(f"✓ Found {len(provider_year_map)} providers with parquet files:")
for pname, pid in provider_options:
    years = sorted(provider_year_map[pid].keys())
    print(f"  {pname} (ID: {pid}): years {years}")

# Create dropdowns
output_parquet_load = widgets.Output()

# Provider dropdown
provider_dropdown = widgets.Dropdown(
    options=[(name, pid) for name, pid in provider_options],
    description='Provider:',
    disabled=False
)

# Year dropdown - will be populated based on selected provider
year_dropdown = widgets.Dropdown(
    options=[],
    description='Year:',
    disabled=False
)

def update_year_options(change=None):
    """Update available years when provider changes"""
    selected_provider_id = provider_dropdown.value
    available_years = sorted(provider_year_map[selected_provider_id].keys())
    year_dropdown.options = available_years
    if available_years:
        year_dropdown.value = available_years[-1]  # Default to most recent year

def load_selected_parquet(change=None):
    """Load the selected parquet file and update df_fixed_monthly"""
    global df_fixed_monthly, parquet_path
    output_parquet_load.clear_output(wait=True)
    with output_parquet_load:
        try:
            selected_provider_id = provider_dropdown.value
            selected_year = year_dropdown.value
            
            if selected_provider_id not in provider_year_map or selected_year not in provider_year_map[selected_provider_id]:
                print(f"✗ No parquet file found for Provider {selected_provider_id}, Year {selected_year}")
                return
            
            parquet_path = provider_year_map[selected_provider_id][selected_year]
            
            # Load parquet file
            df_parquet = pd.read_parquet(parquet_path)
            print(f"✓ Parquet file loaded: {len(df_parquet)} records")
            print(f"  Provider: {provider_dropdown.options}")
            print(f"  Year: {selected_year}")
            print(f"  Path: {parquet_path}")
            
            # Map parquet columns to expected format
            # month_label -> month, contract_duration -> contract_type
            df_fixed_monthly = df_parquet[[
                'month_label', 'provider', 'contract_name', 'contract_duration',
                'elec_piek_per_kwh', 'elec_dal_per_kwh'
            ]].copy()
            
            df_fixed_monthly.columns = [
                'month', 'provider', 'contract_name', 'contract_type',
                'piek_tariff', 'dal_tariff'
            ]
            
            # Extract year from month string for filtering
            df_fixed_monthly['year_extracted'] = df_fixed_monthly['month'].str.extract(r'(\d{4})')[0].astype(int)
            
            print(f"\n✓ Fixed/variable contract data loaded: {len(df_fixed_monthly)} records")
            print(f"  Providers in data: {df_fixed_monthly['provider'].nunique()}")
            print(f"  Sample providers: {df_fixed_monthly['provider'].unique()[:3].tolist()}")
            print(f"  Months: {df_fixed_monthly['month'].nunique()}")
            print(f"  Unique contracts: {df_fixed_monthly['contract_name'].nunique()}")
            
            # Clear cached data to force rebuild with new parquet
            series_cache.clear()
            print(f"\n✓ Cleared series cache for visualization rebuild")
            
        except Exception as e:
            import traceback
            print(f"✗ Error loading parquet file: {e}")
            print(f"  Traceback: {traceback.format_exc()}")
            df_fixed_monthly = pd.DataFrame()

# Set up observers
provider_dropdown.observe(update_year_options, names='value')
provider_dropdown.observe(load_selected_parquet, names='value')
year_dropdown.observe(load_selected_parquet, names='value')

# Initialize year dropdown with options for the first provider
update_year_options()

# Display dropdowns and load initial selection
print("\n📊 SELECT PROVIDER AND YEAR TO LOAD DATA:\n")
controls_display = widgets.HBox([provider_dropdown, year_dropdown])
display(controls_display)
display(output_parquet_load)

# Load initially selected parquet
load_selected_parquet()

✓ Found 56 providers with parquet files:
  AllureNRG (ID: 1): years [2023, 2024, 2025, 2026]
  ANWB Energie (ID: 2): years [2023, 2024, 2025, 2026]
  Audax Renewables (ID: 3): years [2023, 2024, 2025, 2026]
  Budget Energie (ID: 4): years [2023, 2024, 2025, 2026]
  Cleanenergy (ID: 5): years [2023, 2024, 2025, 2026]
  CNS (ID: 6): years [2023, 2024, 2025, 2026]
  Coolblue Energie (ID: 7): years [2023, 2024, 2025, 2026]
  DELTA energie (ID: 8): years [2023, 2024, 2025, 2026]
  DGB energie (ID: 9): years [2023, 2024]
  DigiWatt Energie (ID: 10): years [2023]
  DVEP Energie (ID: 11): years [2023]
  EasyEnergy (ID: 12): years [2023, 2024, 2025]
  ELIX (ID: 13): years [2023, 2024, 2025, 2026]
  Eneco (ID: 14): years [2023, 2024, 2025, 2026]
  Eneco Zakelijk (ID: 15): years [2024, 2025, 2026]
  Energie.VanOns (ID: 16): years [2023, 2024, 2025, 2026]
  Energiedirect.nl (ID: 17): years [2023, 2024, 2025, 2026]
  Energiek (ID: 18): years [2024, 2025, 2026]
  Energyhouse (ID: 19): years [2023, 2

Output()

## 5. Persistent Parquet Cache & Resolution-Aware Data Retrieval

In [15]:
from pathlib import Path

cache_dir = Path(r'c:\Users\20203525\Documents\2025 2026\WB4U\wb4y-webscraper\Visualizations\cache')
cache_dir.mkdir(exist_ok=True)

def load_or_build_parquet(path, build_fn):
    """Load parquet from disk or build + save if missing"""
    if path.exists():
        return pd.read_parquet(path)
    else:
        df = build_fn()
        df.to_parquet(path, index=False)
        return df

def build_spot_15m():
    df = df_entsoe[['price_eur_per_kwh']].copy()
    df = df.reset_index()
    df.columns = ['datetime', 'spot_eur_per_kwh']
    df['datetime'] = pd.to_datetime(df['datetime']).dt.tz_convert('Europe/Amsterdam')
    return df

def build_spot_hourly():
    df = build_spot_15m()
    # Use strftime to avoid DST ambiguity in floor
    df['datetime_str'] = df['datetime'].dt.strftime('%Y-%m-%d %H:00:00')
    df['datetime_hour'] = pd.to_datetime(df['datetime_str']).dt.tz_localize('Europe/Amsterdam', ambiguous='NaT')
    df = df.dropna(subset=['datetime_hour'])
    agg = df.groupby('datetime_hour')['spot_eur_per_kwh'].mean().reset_index()
    agg.columns = ['datetime_hour', 'spot_hourly_mean_eur_per_kwh']
    return agg

def build_spot_daily():
    hourly_df = load_or_build_parquet(cache_dir / 'spot_hourly.parquet', build_spot_hourly)
    hourly_df['date'] = pd.to_datetime(hourly_df['datetime_hour']).dt.date
    agg = hourly_df.groupby('date')['spot_hourly_mean_eur_per_kwh'].mean().reset_index()
    agg.columns = ['date', 'spot_daily_mean_eur_per_kwh']
    return agg

def build_fv_segments():
    rows = []
    for idx, row in df_fixed_monthly.iterrows():
        month_str = row['month']
        month_date = pd.to_datetime(month_str)
        year, month_num = month_date.year, month_date.month
        month_start = pd.Timestamp(f'{year}-{month_num:02d}-01', tz='Europe/Amsterdam')
        if month_num == 12:
            month_end = pd.Timestamp(f'{year+1}-01-01', tz='Europe/Amsterdam')
        else:
            month_end = pd.Timestamp(f'{year}-{month_num+1:02d}-01', tz='Europe/Amsterdam')
        rows.append({
            'provider': row['provider'],
            'contract_name': row['contract_name'],
            'contract_type': row['contract_type'],
            'month_start': month_start,
            'month_end': month_end,
            'piek_price_eur_per_kwh': row['piek_tariff'],
            'dal_price_eur_per_kwh': row['dal_tariff']
        })
    return pd.DataFrame(rows)

def build_margin_segments():
    rows = []
    for provider, data in dynamic_providers.items():
        rows.append({
            'provider': data['provider_name'],
            'valid_from': pd.Timestamp('2025-01-01', tz='Europe/Amsterdam'),
            'valid_to': pd.Timestamp('2026-12-31', tz='Europe/Amsterdam'),
            'margin_eur_per_kwh': data['margin']
        })
    return pd.DataFrame(rows)

def build_energy_tax():
    return pd.DataFrame({
        'year': [2025, 2026],
        'energy_tax_eur_per_kwh': [ELEC_TAX, ELEC_TAX]
    })

spot_15m = load_or_build_parquet(cache_dir / 'spot_15m.parquet', build_spot_15m)
spot_hourly = load_or_build_parquet(cache_dir / 'spot_hourly.parquet', build_spot_hourly)
spot_daily = load_or_build_parquet(cache_dir / 'spot_daily.parquet', build_spot_daily)
fv_segments = load_or_build_parquet(cache_dir / 'fv_month_segments.parquet', build_fv_segments)
margin_segments = load_or_build_parquet(cache_dir / 'margin_segments.parquet', build_margin_segments)
energy_tax_year = load_or_build_parquet(cache_dir / 'energy_tax_year.parquet', build_energy_tax)

print("✓ Cache infrastructure ready")
print(f"  Spot 15m: {len(spot_15m)} rows")
print(f"  Spot hourly: {len(spot_hourly)} rows")
print(f"  Spot daily: {len(spot_daily)} rows")
print(f"  Fixed/variable segments: {len(fv_segments)} rows")
print(f"  Margin segments: {len(margin_segments)} rows")

✓ Cache infrastructure ready
  Spot 15m: 38388 rows
  Spot hourly: 9596 rows
  Spot daily: 401 rows
  Fixed/variable segments: 2502 rows
  Margin segments: 19 rows


In [20]:
series_cache = {}

def get_series(contract_type, provider, contract_name, start, end, resolution):
    """
    Retrieve cost timeseries at specified resolution for selected window only.
    Caches results to avoid recomputation. Never upsamples globally.
    """
    cache_key = (contract_type, provider, contract_name, str(start), str(end), resolution)
    if cache_key in series_cache:
        return series_cache[cache_key]
    
    # Handle timezone-aware or naive timestamps
    start_ts = pd.Timestamp(start)
    end_ts = pd.Timestamp(end)
    if start_ts.tz is None:
        start_ts = start_ts.tz_localize('Europe/Amsterdam')
    if end_ts.tz is None:
        end_ts = end_ts.tz_localize('Europe/Amsterdam')
    
    if contract_type == 'dynamic':
        # Dynamic: spot price + margin + tax
        if resolution == '15min':
            df = spot_15m[(spot_15m['datetime'] >= start_ts) & (spot_15m['datetime'] <= end_ts)].copy()
            margin = margin_segments[margin_segments['provider'] == provider]['margin_eur_per_kwh'].iloc[0]
            year = df['datetime'].iloc[0].year if len(df) > 0 else start_ts.year
            tax = energy_tax_year[energy_tax_year['year'] == year]['energy_tax_eur_per_kwh'].iloc[0]
            df['cost_eur_per_kwh'] = df['spot_eur_per_kwh'] * (1 + VAT_RATE) + margin + tax
            result = df[['datetime', 'cost_eur_per_kwh']].copy()
        elif resolution == 'hourly':
            df = spot_hourly[(spot_hourly['datetime_hour'] >= start_ts) & (spot_hourly['datetime_hour'] <= end_ts)].copy()
            margin = margin_segments[margin_segments['provider'] == provider]['margin_eur_per_kwh'].iloc[0]
            year = df['datetime_hour'].iloc[0].year if len(df) > 0 else start_ts.year
            tax = energy_tax_year[energy_tax_year['year'] == year]['energy_tax_eur_per_kwh'].iloc[0]
            df['cost_eur_per_kwh'] = df['spot_hourly_mean_eur_per_kwh'] * (1 + VAT_RATE) + margin + tax
            result = df[['datetime_hour', 'cost_eur_per_kwh']].copy()
            result.columns = ['datetime', 'cost_eur_per_kwh']
        else:  # daily
            df = spot_daily[(spot_daily['date'] >= start_ts.date()) & (spot_daily['date'] <= end_ts.date())].copy()
            margin = margin_segments[margin_segments['provider'] == provider]['margin_eur_per_kwh'].iloc[0]
            year = pd.Timestamp(df['date'].iloc[0]).year if len(df) > 0 else start_ts.year
            tax = energy_tax_year[energy_tax_year['year'] == year]['energy_tax_eur_per_kwh'].iloc[0]
            df['cost_eur_per_kwh'] = df['spot_daily_mean_eur_per_kwh'] * (1 + VAT_RATE) + margin + tax
            df['datetime'] = pd.to_datetime(df['date'], utc=True).dt.tz_convert('Europe/Amsterdam')
            result = df[['datetime', 'cost_eur_per_kwh']].copy()
    
    else:  # fixed_variable
        if resolution == '15min':
            times = pd.date_range(start_ts, end_ts, freq='15min', tz='Europe/Amsterdam')
            costs = []
            for ts in times:
                is_peak = not is_dal_hour(ts)
                segment = fv_segments[(fv_segments['provider'] == provider) & 
                                     (fv_segments['contract_name'] == contract_name) &
                                     (fv_segments['month_start'] <= ts) & 
                                     (ts < fv_segments['month_end'])]
                if len(segment) > 0:
                    piek_price = segment.iloc[0]['piek_price_eur_per_kwh']
                    dal_price = segment.iloc[0]['dal_price_eur_per_kwh']
                    if pd.isna(dal_price):
                        dal_price = piek_price
                    price = piek_price if is_peak else dal_price
                    costs.append(price)
                else:
                    costs.append(np.nan)
            result = pd.DataFrame({'datetime': times, 'cost_eur_per_kwh': costs})
        
        elif resolution == 'hourly':
            times = pd.date_range(start_ts, end_ts, freq='1h', tz='Europe/Amsterdam')
            costs = []
            for ts in times:
                is_peak = not is_dal_hour(ts)
                segment = fv_segments[(fv_segments['provider'] == provider) & 
                                     (fv_segments['contract_name'] == contract_name) &
                                     (fv_segments['month_start'] <= ts) & 
                                     (ts < fv_segments['month_end'])]
                if len(segment) > 0:
                    piek_price = segment.iloc[0]['piek_price_eur_per_kwh']
                    dal_price = segment.iloc[0]['dal_price_eur_per_kwh']
                    if pd.isna(dal_price):
                        dal_price = piek_price
                    price = piek_price if is_peak else dal_price
                    costs.append(price)
                else:
                    costs.append(np.nan)
            result = pd.DataFrame({'datetime': times, 'cost_eur_per_kwh': costs})
        
        else:  # daily
            times = pd.date_range(start_ts, end_ts, freq='1d', tz='Europe/Amsterdam')
            costs = []
            for ts in times:
                segment = fv_segments[(fv_segments['provider'] == provider) & 
                                     (fv_segments['contract_name'] == contract_name) &
                                     (fv_segments['month_start'] <= ts) & 
                                     (ts < fv_segments['month_end'])]
                if len(segment) > 0:
                    piek = segment.iloc[0]['piek_price_eur_per_kwh']
                    dal = segment.iloc[0]['dal_price_eur_per_kwh']
                    if pd.isna(dal):
                        dal = piek
                    price = (piek + dal) / 2
                    costs.append(price)
                else:
                    costs.append(np.nan)
            result = pd.DataFrame({'datetime': times, 'cost_eur_per_kwh': costs})
    
    series_cache[cache_key] = result
    return result

print("✓ get_series() function ready (caches results in series_cache)")    

✓ get_series() function ready (caches results in series_cache)


## 6. Example Comparison: Frank Energie (Dynamic) vs Vattenfall (Variable)

In [21]:
# Helper function for dal hour detection and contract validation
def is_dal_hour(ts):
    """Check if timestamp is during DAL (night) hours"""
    hour = ts.hour
    return hour in DAL_HOURS

def find_complete_contracts_by_year(year=None):
    """Find contracts with both piek and dal for all months of the specified year"""
    if df_fixed_monthly.empty:
        return pd.DataFrame()
    
    # If no year specified, use the most recent year in the data
    if year is None:
        year_str = df_fixed_monthly['month'].iloc[0] if len(df_fixed_monthly) > 0 else ''
        try:
            year = int(year_str.split()[-1])
        except:
            return pd.DataFrame()
    
    year_str = str(year)
    months_in_year = df_fixed_monthly[df_fixed_monthly['month'].str.contains(year_str)]['month'].unique()
    
    results = []
    
    for provider in df_fixed_monthly['provider'].unique():
        for contract_name in df_fixed_monthly[df_fixed_monthly['provider'] == provider]['contract_name'].unique():
            # Get all records for this provider/contract in the specified year
            contracts = df_fixed_monthly[
                (df_fixed_monthly['provider'] == provider) &
                (df_fixed_monthly['contract_name'] == contract_name) &
                (df_fixed_monthly['month'].str.contains(year_str))
            ]
            
            # Check if all months have both piek and dal
            if len(contracts) == len(months_in_year):  # Has all months
                if (contracts['piek_tariff'].notna().all() and contracts['dal_tariff'].notna().all()):
                    avg_piek = contracts['piek_tariff'].mean()
                    avg_dal = contracts['dal_tariff'].mean()
                    # Check that dal is actually different from piek (not fallback)
                    if avg_dal > 0 and abs(avg_dal - avg_piek) > 0.001:
                        results.append({
                            'provider': provider,
                            'contract_name': contract_name,
                            'avg_piek': avg_piek,
                            'avg_dal': avg_dal,
                            'months_count': len(contracts),
                            'year': year
                        })
    
    return pd.DataFrame(results) if results else pd.DataFrame()

# Find suitable contracts for available years
if not df_fixed_monthly.empty:
    # Get all unique years in the data
    df_fixed_monthly['year_extracted'] = df_fixed_monthly['month'].str.extract(r'(\d{4})')[0].astype(int)
    available_years = sorted(df_fixed_monthly['year_extracted'].unique(), reverse=True)
    
    print("✓ Contract validation complete")
    print(f"  Available years: {available_years}")
    
    # Find complete contracts for each year
    for year in available_years:
        complete_contracts = find_complete_contracts_by_year(year)
        print(f"\n  Year {year}: {len(complete_contracts)} contracts with full piek+dal coverage")
        if len(complete_contracts) > 0 and len(complete_contracts) <= 5:
            for idx, row in complete_contracts.iterrows():
                print(f"    - {row['provider']}: {row['contract_name']} (piek: €{row['avg_piek']:.4f}, dal: €{row['avg_dal']:.4f})")
    
    # Use the most recent year's contracts
    if available_years:
        complete_contracts_current = find_complete_contracts_by_year(available_years[0])
    else:
        complete_contracts_current = pd.DataFrame()
else:
    print("✗ No contract data loaded")
    complete_contracts_current = pd.DataFrame()


✓ Contract validation complete
  Available years: [2025]

  Year 2025: 0 contracts with full piek+dal coverage


In [22]:
# Time period selector with year and month toggles
period_options = {'1 Day': 1, '1 Week': 7, '1 Month': 30, '1 Year': 365}
dropdown_period = widgets.Dropdown(
    options=list(period_options.keys()),
    value='1 Month',
    description='Time Period:'
)

# Ensure year_extracted column exists
if not df_fixed_monthly.empty and 'year_extracted' not in df_fixed_monthly.columns:
    df_fixed_monthly['year_extracted'] = df_fixed_monthly['month'].str.extract(r'(\d{4})')[0].astype(int)

# Year selector - dynamically populate with available years
available_years_list = sorted(df_fixed_monthly['year_extracted'].unique(), reverse=True) if not df_fixed_monthly.empty else ['2026']
dropdown_year = widgets.Dropdown(
    options=[str(y) for y in available_years_list],
    value=str(available_years_list[0]) if available_years_list else '2026',
    description='Year:'
)